[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DeutscheAktuarvereinigung/.github/blob/main/traffic/traffic_analysis.ipynb)

# Traffic Analysis – DeutscheAktuarvereinigung

Dieses Notebook lädt die aktuellste GitHub-Traffic-CSV aus Google Drive und analysiert sie interaktiv mit Altair.

In [4]:
# Setup - importing libraries
import pandas as pd
import altair as alt
import glob
import os
import requests # Added requests for fetching files from URLs

In [5]:
# Setup - Determinining run environment
try:
    import google.colab
    environment = 'colab'
    print("Running in Google Colab")
    # Further Colab-specific code here
except ImportError:
    print("Running in a local environment")
    environment = 'local'
    # Further local environment-specific code here

Running in Google Colab


In [6]:
# Aktuellste Traffic-CSV automatisch ermitteln von GitHub
if environment == 'colab':
  GITHUB_API_URL = 'https://api.github.com/repos/DeutscheAktuarvereinigung/.github/contents/traffic'

  response = requests.get(GITHUB_API_URL)
  response.raise_for_status() # Raise an exception for HTTP errors
  files_data = response.json()

  csv_files_info = []
  for file in files_data:
      if file['type'] == 'file' and file['name'].startswith('github_traffic_dav_') and file['name'].endswith('.csv'):
          csv_files_info.append(file)

  if not csv_files_info:
      raise FileNotFoundError(f'Keine github_traffic_dav_*.csv in der GitHub-URL {GITHUB_API_URL} gefunden.')

else:
  DRIVE_BASE = '.'
  csv_files = sorted(glob.glob(os.path.join(DRIVE_BASE, 'github_traffic_dav_*.csv')), reverse=True)

# Sort by name to get the 'latest' assuming names contain date/version info
csv_files_info.sort(key=lambda x: x['name'], reverse=True)

# Get the download URL of the latest CSV file
OUTPUT_CSV_PATH = csv_files_info[0]['download_url']
print(f'Verwende aktuellste CSV von GitHub: {OUTPUT_CSV_PATH}')

Verwende aktuellste CSV von GitHub: https://raw.githubusercontent.com/DeutscheAktuarvereinigung/.github/main/traffic/github_traffic_dav_2026-05-12.csv


In [7]:
# CSV laden
try:
    df_from_csv = pd.read_csv(OUTPUT_CSV_PATH, sep=';')
    # Convert 'Datum' column to datetime objects
    df_from_csv['Datum'] = pd.to_datetime(df_from_csv['Datum'])
    print(f'CSV loaded successfully from: {OUTPUT_CSV_PATH}')
except FileNotFoundError:
    print(f'Error: The file {OUTPUT_CSV_PATH} was not found. Please ensure the data collection cell was executed and the file was saved.')
    raise

# Re-create df_clean for charting purposes, using the data loaded from CSV
df_clean_from_csv = df_from_csv[df_from_csv['Datum'] > (df_from_csv['Datum'].max() - pd.Timedelta(days=14))]

# Define metrics list
metrics = ['Views', 'Clones', 'Unique Views', 'Unique Clones']

display(df_clean_from_csv.head())
print('Metrics for charting:', metrics)

CSV loaded successfully from: https://raw.githubusercontent.com/DeutscheAktuarvereinigung/.github/main/traffic/github_traffic_dav_2026-05-12.csv


,Repository,Datum,Views,Clones,Unique Views,Unique Clones
0,.github,2026-05-10,0,10,0,7
1,.github,2026-05-09,1,14,1,9
2,.github,2026-05-08,0,18,0,10
3,.github,2026-05-07,1,16,1,11
4,.github,2026-05-06,0,13,0,8


Metrics for charting: ['Views', 'Clones', 'Unique Views', 'Unique Clones']


In [12]:
# Summarize repositories by total views and unique clones over the two-week period
repo_views_summary = df_clean_from_csv.groupby('Repository').agg(
    Views=('Views', 'sum'),
    Unique_Clones=('Unique Clones', 'sum')
).reset_index()

# Sort by views in descending order
repo_views_summary = repo_views_summary.sort_values(by='Views', ascending=False)

print("Repositories summarized by total views and unique clones over the last two weeks:")
display(repo_views_summary)

Repositories summarized by total views and unique clones over the last two weeks:


,Repository,Views,Unique_Clones
23,insurance_scr_data,59,4
3,2025_CADS_Immersion_Best_Notebooks,49,4
22,claim_frequency,35,4
15,Python_fuer_Aktuare,34,4
4,2026_CADS_Completion_Best_Notebooks,33,4
20,WorkingGroup_Synthetic_Data,16,4
14,Mortality_Modeling,10,4
12,GenAI_Beyond_the_Basics,10,5
21,WorkingGroup_eXplainableAI_Notebooks,10,5
1,2024_CADS_Immersion_Best_Notebooks,9,5


In [8]:
if environment == 'colab':

  import altair as alt

  # Create a selection for the metric dropdown
  metric_selection = alt.selection_point(
      fields=['Metric'],
      bind=alt.binding_select(options=metrics, name='Select Metric '),
      value=metrics[0],
      name='metric_param'
  )

  # Create a multi-selection for repositories
  repository_selection = alt.selection_point(
      fields=['Repository'],
      bind=alt.binding_select(options=df_clean_from_csv['Repository'].unique().tolist(), name='Select Repository '),
      name='repo_select'
  )

  # Create the base chart
  base = alt.Chart(df_clean_from_csv).properties(
      title='GitHub Traffic Trends by Repository'
  ).add_params(
      metric_selection,
      repository_selection
  ).transform_fold(
      metrics,
      as_=['Metric', 'Value']
  ).transform_filter(
      metric_selection
  ).transform_filter(
      repository_selection
  )

  # Create the line chart
  chart = base.mark_line(point=True).encode(
      x=alt.X('Datum:T', title='Date'),
      y=alt.Y('Value:Q', title='Metric Value', stack=None),
      color=alt.Color('Repository:N', title='Repository'),
      tooltip=[
          alt.Tooltip('Repository:N'),
          alt.Tooltip('Datum:T', title='Date'),
          alt.Tooltip('Value:Q', title='Metric Count', format='.0f')
      ]
  ).interactive()

  chart

alt.Chart(...)